# 05 · Self-RAG AutoML — From Natural Language to Metalens

This notebook runs the full LangGraph workflow. You give it a
natural-language requirement, it parses the requirement, retrieves
relevant background from a markdown corpus, generates design
parameters, runs RCWA, optimizes, reflects on the result, and (when
Strehl ≥ target) produces a GDS file.

## Prerequisites

- An LLM provider configured in `.env`:
  - `LLM_DEFAULT_PROVIDER=ollama` and a local Ollama with a model
    (`qwen2.5:7b` is the project default for tool-calling), **or**
  - `LLM_DEFAULT_PROVIDER=groq` and `GROQ_API_KEY=...`, **or**
  - `LLM_DEFAULT_PROVIDER=anthropic` and `ANTHROPIC_API_KEY=...`.
- The analytical backend is enough — vendor metabox3 only if you
  want real RCWA inside the loop.

If your provider isn't ready, this notebook will warn but won't error.

In [1]:
import sys, asyncio, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from metaopticsai.configs.settings import settings
from metaopticsai.llm.registry import get_provider, available_providers

print(f"Default provider: {settings.llm.default_provider}")
print(f"Available providers: {available_providers()}")

provider = get_provider()
print(f"Provider OK?       {provider.is_available()}")

Default provider: ollama
Available providers: ['anthropic', 'groq', 'ollama']
Provider OK?       True


## 1 · Build the controller

`MetaOpticsController.build()` is the single composition root — store,
backends, tools, knowledge base, grader, workflow. One call wires it
all up.

In [2]:
from metaopticsai.orchestration.controller import MetaOpticsController

controller = MetaOpticsController.build()
print(f"Tools registered: {len(list(controller.tools.names()))}")
print(f"KB ready:         {controller.kb.is_ready()}")
print(f"Backends:         {[b.name for b in controller.backends]}")

d:\metacode\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


AttributeError: 'WorkflowConfig' object has no attribute 'grader_threshold'

## 2 · Run the workflow

The graph: `parse → retrieve → grade → generate → simulate → optimize
→ evaluate → reflect → (loop or finalize)`.

Long-running cell — expect 30s-3min depending on provider speed and
number of self-reflection iterations.

In [3]:
REQUIREMENT = (
    "Design a TiO2 metalens for 532 nm wavelength, focal length 100 µm, "
    "diameter 50 µm, Strehl ratio at least 0.85, polarization-insensitive."
)

result = asyncio.run(
    controller.design_metalens(REQUIREMENT, target_strehl=0.85)
)

NameError: name 'controller' is not defined

## 3 · Inspect every stage of the workflow output

In [ ]:
print("── Parsed requirements ──")
print(json.dumps(result.get("parsed"), indent=2))

In [ ]:
print("── Retrieved docs (Self-RAG) ──")
for d in result.get("retrieved_docs", []):
    print(f"  · {d['source']}  (score {d.get('score', 0):.2f})")

In [ ]:
print("── Generated design parameters ──")
print(json.dumps(result.get("design_params"), indent=2))

In [ ]:
print("── RCWA sweep metrics ──")
print(json.dumps(result.get("sweep_metrics"), indent=2))

In [ ]:
print("── Optimization outcome ──")
print(f"Achieved Strehl: {result.get('achieved_strehl', 0.0):.3f}")
print(f"Decision history:")
for d in result.get("decision_history", []):
    print(f"  · {d['decision']:<14} {d.get('rationale', '')[:80]}")

In [ ]:
print("── Final summary ──\n")
print(result.get("summary", "(no summary)"))

## 4 · Where are the artifacts?

Every handle stays in the in-process store. Inspect them all:

In [ ]:
for art in controller.store.list():
    print(f"  {art['kind']:<16} {art['handle']}  {art['metadata']}")

## What just happened

The workflow ran several **self-reflection loops** internally — each
time the reflect node looks at Strehl/coverage/transmission and decides
whether to *accept*, *re-optimize*, *re-sweep*, or *re-retrieve*. The
decision_history field tells you the trajectory.

For production runs, submit this as a background job via
`submit_job` (returns a `job_id`) and poll with `get_job`, so the
MCP client thread stays responsive.